<a href="https://colab.research.google.com/github/itsmeyessir/llm-decon/blob/main/llm-eval-harness/baseline_generation_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Runtime Environment & Background Service Initialization

Evaluating Large Language Models locally requires managing both system dependencies and execution state within Colab's ephemeral runtime.

### Architectural Considerations
* **Daemon Isolation:** We deploy the Ollama runtime binary directly to the host environment. To prevent blocking the Jupyter kernel event loop during execution, the service daemon is initialized asynchronously via `subprocess.Popen`.
* **Process Warmup:** A 5-second execution pause is introduced post-spawn to allow the background HTTP socket listener (`127.0.0.1:11434`) to bind before downstream API requests are dispatched.

In [36]:
# -------------------------------------------------------------------
# System Dependency Injection & Daemon Initialization
# -------------------------------------------------------------------

# Provision OS-level dependencies for binary extraction (quiet mode)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

# Fetch and execute the Ollama deployment script via standard streams
!curl -fsSL https://ollama.com/install.sh | sh

# Provision Python runtime packages
!pip install -q ragas langchain-community langchain-ollama langchain-huggingface pandas

import subprocess
import time
import urllib.request
import urllib.error

def initialize_ollama_daemon():
    """
    Forks the Ollama server process from the main Jupyter kernel execution thread.
    Utilizes subprocess.Popen to prevent blocking the synchronous notebook runtime.
    """
    print("[SYSTEM] Initializing Ollama daemon process...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

def wait_for_socket(url="http://127.0.0.1:11434/", timeout=30, poll_interval=0.5):
    """
    Actively polls the HTTP daemon endpoint until a verified HTTP 200 response
    is received, guaranteeing downstream cells only run on verified server readiness.
    """
    print(f"[SYSTEM] Polling daemon socket at {url}...")
    start_time = time.time()

    while time.time() - start_time < timeout:
        try:
            with urllib.request.urlopen(url, timeout=1) as response:
                if response.status == 200:
                    elapsed = round(time.time() - start_time, 2)
                    print(f"[SYSTEM] Verified socket connection (HTTP 200) in {elapsed}s.")
                    print("[SYSTEM] Runtime environment fully provisioned and operational.")
                    return True
        except (urllib.error.URLError, ConnectionRefusedError, OSError):
            time.sleep(poll_interval)

    raise RuntimeError(f"[CRITICAL] Daemon failed to respond on {url} within {timeout} seconds.")

# Execution Pipeline
initialize_ollama_daemon()
wait_for_socket()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[SYSTEM] Initializing Ollama daemon process...
[SYSTEM] Polling daemon socket at http://127.0.0.1:11434/...
[SYSTEM] Verified socket connection (HTTP 200) in 0.0s.
[SYSTEM] Runtime environment fully provisioned and operational.


True

## 2. Model Provisioning & Compute Constraints

The evaluation harness requires an engine capable of structured output parsing and multi-step reasoning to serve as the metric judge.

### Hardware & Allocation Strategy
* **Target Hardware:** Google Colab T4 GPU Instance (~15 GB VRAM).
* **Selected Model:** `Llama 3.1 8B Instruct` (4-bit quantized via GGUF).
* **Rationale:** Small Language Models (SLMs) under 7B parameters frequently fail when enforcing strict JSON schema constraints under system prompts. The 8B variant provides the necessary instruction-following threshold while remaining safely within the T4 VRAM footprint during concurrency.

In [37]:
# -------------------------------------------------------------------
# Model Provisioning, Idempotency & Manifest Registration
# -------------------------------------------------------------------
import json
import subprocess
import urllib.request
import urllib.error

def get_registered_models(daemon_url="http://127.0.0.1:11434/api/tags"):
    """
    Queries the local Ollama REST API to retrieve the current inventory of pulled models.
    Returns a dictionary mapping model names to their size in gigabytes.
    """
    try:
        req = urllib.request.Request(daemon_url)
        with urllib.request.urlopen(req, timeout=5) as response:
            if response.status == 200:
                payload = json.loads(response.read().decode("utf-8"))
                models = payload.get("models", [])
                inventory = {}
                for m in models:
                    name = m.get("name", "")
                    size_gb = round(m.get("size", 0) / (1024**3), 2)
                    inventory[name] = size_gb
                return inventory
    except Exception as e:
        print(f"[WARN] Failed to query daemon inventory: {str(e)}")
        return {}


def ensure_model_provisioned(target_model="llama3.1:8b"):
    """
    Idempotent model provisioner:
    1. Checks if model exists locally.
    2. Pulls model if absent, capturing process exit codes.
    3. Verifies post-pull registration and reports exact system state.
    """
    print(f"[SYSTEM] Inspecting local inventory for model '{target_model}'...")
    inventory = get_registered_models()

    # Fuzzy match tag (handles 'llama3.1:8b' vs 'llama3.1:latest')
    matched_name = next((name for name in inventory if target_model in name or name in target_model), None)

    if matched_name:
        size = inventory[matched_name]
        print(f"[SYSTEM] Local inventory hit: Model '{matched_name}' ({size} GB) is ready. Skipping download.")
        return True

    # Model not found -> Initiate pull sequence
    print(f"[SYSTEM] Model '{target_model}' not found in local inventory. Initiating pull sequence...")

    try:
        # Stream pull output directly to notebook standard streams
        process = subprocess.run(
            ["ollama", "pull", target_model],
            check=True,
            text=True
        )

        # Verify post-pull state
        updated_inventory = get_registered_models()
        verified_match = next((name for name in updated_inventory if target_model in name or name in target_model), None)

        if verified_match:
            size = updated_inventory[verified_match]
            print(f"[SYSTEM] Provisioning successful: Model '{verified_match}' ({size} GB) verified in manifest.")
            return True
        else:
            raise RuntimeError(f"Pull command completed, but '{target_model}' is missing from daemon manifest.")

    except subprocess.CalledProcessError as e:
        print(f"[CRITICAL] Model pull process failed with exit code {e.returncode}.")
        raise RuntimeError(f"Failed to pull model '{target_model}'. Check network connection or model tag.") from e
    except Exception as e:
        print(f"[CRITICAL] Provisioning pipeline encountered an error: {str(e)}")
        raise e


# Execute idempotent model provisioning
ensure_model_provisioned("llama3.1:8b")

[SYSTEM] Inspecting local inventory for model 'llama3.1:8b'...
[SYSTEM] Local inventory hit: Model 'llama3.1:8b' (4.58 GB) is ready. Skipping download.


True

## 3. Ground Truth Data Alignment

To evaluate generative outputs systematically, we decouple the retrieval pipeline from the generator logic by defining a static test vector.

### Data Schema
The evaluation dataset requires four primary fields:
1. `question`: The input prompt passed to the system.
2. `contexts`: The reference knowledge retrieved from the vector store.
3. `ground_truth`: The factual benchmark required for reference metrics.
4. `answer`: The raw output generated by the candidate LLM under test.

In [38]:
# -------------------------------------------------------------------
# Evaluation Data Vector Construction & Schema Validation
# -------------------------------------------------------------------
import pandas as pd
from datasets import Dataset

def construct_and_validate_dataset(eval_data: dict) -> Dataset:
    """
    Constructs a Hugging Face Dataset from raw evaluation vectors while
    enforcing strict schema integrity, dimensional alignment, and type safety.
    """
    required_keys = {"question", "contexts", "ground_truth"}

    print("[SYSTEM] Initiating evaluation dataset schema validation...")

    # 1. Structural Schema Check
    missing_keys = required_keys - set(eval_data.keys())
    if missing_keys:
        raise ValueError(f"[CRITICAL] Dataset schema missing required keys: {missing_keys}")

    # 2. Dimensional Alignment Check across vectors
    lengths = {key: len(eval_data[key]) for key in required_keys}
    unique_lengths = set(lengths.values())

    if len(unique_lengths) > 1:
        raise ValueError(f"[CRITICAL] Dimensional mismatch across dataset columns: {lengths}")

    record_count = list(unique_lengths)[0]
    print(f"[SYSTEM] Dimensional check passed: {record_count} evaluation records detected.")

    # 3. Type Safety & Non-Null Validation
    for idx in range(record_count):
        if not isinstance(eval_data["question"][idx], str) or not eval_data["question"][idx].strip():
            raise TypeError(f"[CRITICAL] Invalid or empty 'question' at index {idx}.")

        if not isinstance(eval_data["contexts"][idx], list) or not eval_data["contexts"][idx]:
            raise TypeError(f"[CRITICAL] 'contexts' at index {idx} must be a non-empty list of string arrays.")

        if not isinstance(eval_data["ground_truth"][idx], str) or not eval_data["ground_truth"][idx].strip():
            raise TypeError(f"[CRITICAL] Invalid or empty 'ground_truth' at index {idx}.")

    print("[SYSTEM] Type safety and non-null constraints verified.")

    # 4. Dataset Instantiation
    dataset = Dataset.from_dict(eval_data)
    print(f"[SYSTEM] Hugging Face Dataset instantiated successfully ({len(dataset)} total records).")

    return dataset

# -------------------------------------------------------------------
# Evaluation Test Vectors (Diverse Challenge Matrix)
# -------------------------------------------------------------------
raw_evaluation_data = {
    "question": [
        "What is the capital of France?",
        "How much memory does the free Colab tier have?",
        "What is the speed of light according to the provided report?",
        "What is the required incident response time for Severity Level 1 outages?",
        "What is the maximum battery capacity of the Quantum Phone X?"
    ],
    "contexts": [
        [
            "Paris is the capital and most populous city of France."
        ],
        [
            "Google Colab provides a free T4 GPU instance with 15GB of VRAM."
        ],
        [
            "In the experimental simulation protocol (Report ID 882), the effective speed of light through the metamaterial buffer is defined as exactly 120,000 km/s."
        ],
        [
            "Enterprise Service Level Agreement (SLA): Severity 1 (Critical Outage) requires initial response within 15 minutes and updates every 30 minutes. Severity 2 (Major Issue) requires initial response within 1 hour."
        ],
        [
            "The Quantum Phone X features a 6.7-inch OLED screen, 12GB of LPDDR5 RAM, and runs on a 4nm processor architecture. Pricing starts at $899."
        ]
    ],
    "ground_truth": [
        "The capital of France is Paris.",
        "The free tier includes 15GB of VRAM.",
        "According to Report ID 882, the speed of light through the metamaterial buffer is 120,000 km/s.",
        "Severity Level 1 outages require an initial response within 15 minutes.",
        "The provided text does not contain information about the battery capacity of the Quantum Phone X."
    ]
}

# Instantiate and validate dataset
eval_dataset = construct_and_validate_dataset(raw_evaluation_data)

# Print verified summary table
df_summary = eval_dataset.to_pandas()
print("\n[SYSTEM] Dataset Summary Preview:")
print(df_summary[["question", "ground_truth"]].to_string(index=True))

[SYSTEM] Initiating evaluation dataset schema validation...
[SYSTEM] Dimensional check passed: 5 evaluation records detected.
[SYSTEM] Type safety and non-null constraints verified.
[SYSTEM] Hugging Face Dataset instantiated successfully (5 total records).

[SYSTEM] Dataset Summary Preview:
                                                                    question                                                                                       ground_truth
0                                             What is the capital of France?                                                                    The capital of France is Paris.
1                             How much memory does the free Colab tier have?                                                               The free tier includes 15GB of VRAM.
2               What is the speed of light according to the provided report?    According to Report ID 882, the speed of light through the metamaterial buffer is 120,000 km/s.
3  W

## 4. Candidate Inference & Deterministic Decoding

Inference parameters must be strictly controlled during evaluation runs to ensure reproducible scoring.

### Execution Control Parameters
* **Temperature ($0.0$):** Disables nucleus sampling to enforce greedy deterministic decoding, ensuring output variance does not contaminate scoring runs.
* **Stress-Testing Adherence:** The prompt template intentionally enforces a multi-sentence output constraint against a single-sentence context block. This introduces controlled length tension to test whether the candidate model adheres strictly to retrieved facts or extrapolates unsupported details.

In [39]:
# -------------------------------------------------------------------
# Candidate Inference & Deterministic Decoding Pipeline
# -------------------------------------------------------------------
import time
from langchain_ollama.llms import OllamaLLM
from datasets import Dataset

def execute_inference_pipeline(dataset: Dataset, target_model: str = "llama3.1:8b", temp: float = 0.0) -> Dataset:
    """
    Executes the inference generation loop over the evaluation dataset.
    Enforces deterministic outputs (temperature=0.0) and wraps execution
    in telemetry and fault-tolerance bounds.
    """
    print(f"[SYSTEM] Initializing inference engine: '{target_model}' (temp={temp})...")

    # Initialize LangChain's Ollama interface targeting the local daemon
    llm = OllamaLLM(model=target_model, temperature=temp)

    generated_responses = []
    total_records = len(dataset)

    start_time = time.time()
    print(f"[SYSTEM] Commencing batch inference over {total_records} records...")

    for idx, record in enumerate(dataset):
        question = record["question"]
        # Contexts are stored as arrays of strings; we extract the primary context block
        context_block = record["contexts"][0]

        # -----------------------------------------------------------
        # The 'Prompt Trap' Constraint
        # -----------------------------------------------------------
        # We explicitly instruct a multi-sentence output against sparse context.
        # This engineered tension forces the LLM to either hallucinate padding
        # or successfully implement boundary adherence.
        prompt = (
            f"Using ONLY the following context, provide a detailed, "
            f"multi-sentence answer to the question: {question}\n"
            f"Context: {context_block}"
        )

        try:
            # Execute blocking inference call
            print(f"  -> [INFERENCE] Processing record {idx + 1}/{total_records}...")
            response = llm.invoke(prompt)
            generated_responses.append(response.strip())
        except Exception as e:
            print(f"[CRITICAL] Inference failed at index {idx}. Reason: {str(e)}")
            # Append a standardized failure token to maintain dataset dimensional alignment
            generated_responses.append("[SYSTEM_INFERENCE_FAILURE]")

    elapsed_time = round(time.time() - start_time, 2)
    print(f"[SYSTEM] Batch inference completed in {elapsed_time}s.")

    # Dataset state mutation: Ensure idempotency by scrubbing legacy columns
    working_dataset = dataset
    if "answer" in working_dataset.column_names:
        print("[SYSTEM] Scrubbing legacy 'answer' column from previous executions...")
        working_dataset = working_dataset.remove_columns(["answer"])

    # Bind the generated array to the dataset
    print("[SYSTEM] Binding generated outputs to evaluation schema...")
    working_dataset = working_dataset.add_column("answer", generated_responses)

    return working_dataset

# Execute the inference pipeline
eval_dataset = execute_inference_pipeline(eval_dataset)

# Preview the injected answers
print("\n[SYSTEM] Generated Output Preview:")
df_preview = eval_dataset.to_pandas()
print(df_preview[["question", "answer"]].head(2).to_string(index=True))

[SYSTEM] Initializing inference engine: 'llama3.1:8b' (temp=0.0)...
[SYSTEM] Commencing batch inference over 5 records...
  -> [INFERENCE] Processing record 1/5...
  -> [INFERENCE] Processing record 2/5...
  -> [INFERENCE] Processing record 3/5...
  -> [INFERENCE] Processing record 4/5...
  -> [INFERENCE] Processing record 5/5...
[SYSTEM] Batch inference completed in 28.67s.
[SYSTEM] Binding generated outputs to evaluation schema...

[SYSTEM] Generated Output Preview:
                                         question                                                                                                                                                                                                                                                                                                                                                                                                                                                answer
0                  What is the capital 

## 5. Automated Evaluation & AST Sanitization

We execute automated scoring using the `Ragas` framework via an "LLM-as-a-Judge" architecture.

### Metrics Computed
* **Faithfulness:** Quantifies factual consistency by extracting claims from the generated answer and checking them against the reference context.
* **Answer Relevancy:** Measures how directly the generated answer addresses the user prompt using embedding space cosine similarity (`all-MiniLM-L6-v2`).

### System Hygiene & Legacy Patching
1. **Dynamic Module Patching:** Overrides broken legacy imports inside `ragas` (`langchain_community.chat_models.vertexai`) by injecting a zero-dependency proxy module into `sys.modules`.
2. **Grammar Enforcement:** Inherits from `ChatOllama` to intercept raw output streams, trimming non-JSON preamble text and repairing trailing array commas using regular expressions before AST evaluation.

In [41]:
# -------------------------------------------------------------------
# Metric Evaluation, AST Sanitization & Execution
# -------------------------------------------------------------------
import warnings
import re
import sys
import time
from types import ModuleType

# Suppress deprecation warnings from upstream framework instability
# to maintain a clean execution log for stakeholders.
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# Architecture Patch: Dependency Decoupling
# -----------------------------------------------------------
# Ragas (v0.2+) contains legacy hardcoded VertexAI imports. To maintain
# a 100% local footprint without bloated Google Cloud PIP dependencies,
# we inject a phantom module into the runtime's sys.modules.
print("[SYSTEM] Deploying Phantom Module Patch for Ragas dependency bypass...")
DummyClass = type("DummyClass", (object,), {})

dummy_chat = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat.ChatVertexAI = DummyClass
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat

try:
    import langchain_community.llms
    langchain_community.llms.VertexAI = DummyClass
except ImportError:
    dummy_llms = ModuleType("langchain_community.llms")
    dummy_llms.VertexAI = DummyClass
    sys.modules["langchain_community.llms"] = dummy_llms

# -----------------------------------------------------------
# Core Imports
# -----------------------------------------------------------
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# -----------------------------------------------------------
# Middleware: Output Sanitization
# -----------------------------------------------------------
class JSONSanitizingOllamaWrapper(ChatOllama):
    """
    Middleware interceptor for LLM outputs. Small parameter models (like 8B)
    occasionally prepend conversational text to JSON outputs or leave trailing
    commas, which fatally crashes the downstream Abstract Syntax Tree (AST) parsers.
    This wrapper sanitizes the raw output buffer before passing it to Ragas.
    """
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        result = super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)
        for generation in result.generations:
            if hasattr(generation, 'message'):
                content = generation.message.content

                start_idx = content.find('{')
                end_idx = content.rfind('}')

                if start_idx != -1 and end_idx != -1:
                    clean_json = content[start_idx : end_idx + 1]
                    # Scrub invalid trailing commas in arrays/objects
                    clean_json = re.sub(r',\s*(?=["}\]])', '', clean_json)
                    generation.message.content = clean_json
        return result

# -----------------------------------------------------------
# Component Initialization & Evaluation Execution
# -----------------------------------------------------------
print("[SYSTEM] Initializing Llama 3.1 (8B) Judge configuration...")
base_llm = JSONSanitizingOllamaWrapper(
    model="llama3.1:8b",
    temperature=0,
    format="json",
    system="Respond with raw JSON only. No explanations. Ensure strict schema adherence."
)
evaluator_llm = LangchainLLMWrapper(base_llm)

print("[SYSTEM] Loading local semantic embedding engine (all-MiniLM-L6-v2)...")
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
evaluator_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

print("\n[SYSTEM] Commencing Ragas Evaluation Harness...")
print("[SYSTEM] Processing evaluation vectors. This is compute-intensive and may take a few minutes...")

start_time = time.time()

try:
    results = evaluate(
        dataset=eval_dataset,
        metrics=[faithfulness, answer_relevancy],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings
    )

    elapsed_time = round(time.time() - start_time, 2)
    print(f"\n[SYSTEM] Evaluation matrix computed successfully in {elapsed_time}s.")

    # -----------------------------------------------------------
    # Results Formatting & Telemetry
    # -----------------------------------------------------------
    df = results.to_pandas()

    # Round metrics for cleaner stakeholder presentation
    metric_cols = ['faithfulness', 'answer_relevancy']
    df[metric_cols] = df[metric_cols].round(4)

    # FIX: Use the schema names Ragas dynamically maps to internally
    display_cols = ['user_input', 'response', 'faithfulness', 'answer_relevancy']

    print("\n================ FINAL EVALUATION SCORECARD ================\n")
    display(df[display_cols])
    print("\n============================================================")

except Exception as e:
    print(f"\n[CRITICAL] Evaluation harness execution failed: {str(e)}")

[SYSTEM] Deploying Phantom Module Patch for Ragas dependency bypass...
[SYSTEM] Initializing Llama 3.1 (8B) Judge configuration...
[SYSTEM] Loading local semantic embedding engine (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[SYSTEM] Commencing Ragas Evaluation Harness...
[SYSTEM] Processing evaluation vectors. This is compute-intensive and may take a few minutes...


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]


[SYSTEM] Evaluation matrix computed successfully in 92.69s.

================ FINAL EVALUATION SCORECARD ================



,user_input,response,faithfulness,answer_relevancy
0,What is the capital of France?,The capital of France is Paris. This is eviden...,1.0000,1.0000
1,How much memory does the free Colab tier have?,The free Colab tier has 15GB of video random a...,0.5000,0.7696
2,What is the speed of light according to the pr...,"According to Report ID 882, the speed of light...",0.7500,0.5783
3,What is the required incident response time fo...,"For a Severity Level 1 outage, which is consid...",0.6667,0.9750
4,What is the maximum battery capacity of the Qu...,"Unfortunately, I must inform you that there is...",0.6667,0.0000
